# TrafficVision — Entrenamiento RT-DETR
**Tesis:** Detección y lectura de placas vehiculares ecuatorianas  
**Modelo:** RT-DETR-L (Large) — transformer-based, sin NMS, alta precisión  
**Dataset:** 7,576 imágenes combinadas (global + Ecuador)

> RT-DETR (Real-Time Detection Transformer) supera a YOLO en precisión mAP  
> manteniendo velocidad de inferencia en tiempo real con GPU.

---
### 📋 Orden de ejecución
| Celda | Descripción | Obligatoria |
|-------|-------------|-------------|
| 0 | Anti-desconexión | ✅ Siempre |
| 1 | Verificar GPU | ✅ Siempre |
| 2 | Instalar dependencias | ✅ Siempre |
| 3 | Montar Drive | ✅ Siempre |
| 4 | Verificar datasets | ✅ Siempre |
| 5 | Crear YAML | ✅ Siempre |
| 6 | **Entrenar** (nuevo) | 🔵 Primera vez |
| 7 | **Reanudar** (interrumpido) | 🟡 Si se cortó |
| 8 | Evaluar métricas | ✅ Al finalizar |
| 9 | Exportar modelo | ✅ Al finalizar |

> **RT-DETR vs YOLO11n en T4:**  
> RT-DETR-L requiere ~6-8 GB VRAM (batch=8) y es ~2× más lento por época,  
> pero suele ganar +2-5 pp mAP@50 en datasets de detección de objetos pequeños.

## Celda 0 — Anti-desconexión + monitor de sesión

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELDA 0 — Anti-desconexión + monitor de sesión
# ══════════════════════════════════════════════════════════════════
import time, threading

def heartbeat():
    """Evita la desconexión automática de Colab cada 90 min."""
    clicks = 0
    while True:
        time.sleep(45)
        clicks += 1
        try:
            from google.colab import output
            output.eval_js('document.querySelector("#top-toolbar").click()')
        except Exception:
            pass

t = threading.Thread(target=heartbeat, daemon=True)
t.start()

SESSION_START = time.time()
print('✅ Anti-desconexión activo')
print('Si se interrumpe, usa CELDA 7 (Reanudar) — no pierdas el progreso.')


✅ Anti-desconexión activo
Si se interrumpe, usa CELDA 7 (Reanudar) — no pierdas el progreso.


## Celda 1 — Verificar GPU y RAM disponible

In [ ]:
# CELDA 1 — Verificar GPU y RAM disponible
!nvidia-smi

import torch, psutil, os

cuda_ok = torch.cuda.is_available()
print(f'\n🔧 CUDA disponible: {cuda_ok}')
if cuda_ok:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'   GPU:  {gpu_name}')
    print(f'   VRAM: {gpu_mem:.1f} GB')

    # RT-DETR-L es más pesado que YOLO11n — batch más conservador
    if gpu_mem >= 14:
        rec_batch = 8
        model_rec = 'rtdetr-l.pt'
    elif gpu_mem >= 8:
        rec_batch = 4
        model_rec = 'rtdetr-l.pt'
    else:
        rec_batch = 2
        model_rec = 'rtdetr-l.pt  ⚠️ puede dar OOM, considera rtdetr-m si persiste'

    print(f'   Batch recomendado para RT-DETR-L: {rec_batch}')
    print(f'   Modelo recomendado: {model_rec}')
    print()
    print('   💡 RT-DETR vs YOLO en VRAM (imgsz=640):')
    print('      YOLO11n  batch=16 → ~3-4 GB VRAM')
    print('      RT-DETR-L batch=8 → ~6-8 GB VRAM')
    print('      RT-DETR-L batch=4 → ~4-5 GB VRAM  ← seguro para T4')
else:
    print('❌ Sin GPU — RT-DETR en CPU es inviable para entrenamiento.')
    print('   Solución: Runtime → Cambiar tipo de entorno de ejecución → T4 GPU')

ram = psutil.virtual_memory()
print(f'\n💾 RAM sistema: {ram.available/1024**3:.1f} GB disponibles / {ram.total/1024**3:.1f} GB total')

disk = psutil.disk_usage('/')
print(f'💿 Disco /tmp:   {disk.free/1024**3:.1f} GB libres')


Wed Apr 29 06:59:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Celda 2 — Instalar dependencias

In [ ]:
# CELDA 2 — Instalar dependencias
# ultralytics >= 8.1 incluye soporte completo para RT-DETR
!pip install ultralytics -q

from ultralytics import RTDETR
import ultralytics
print(f'✅ Ultralytics {ultralytics.__version__} instalado')

# Verificar soporte RT-DETR
major, minor = map(int, ultralytics.__version__.split('.')[:2])
if major < 8 or (major == 8 and minor < 1):
    print('⚠️  Versión muy antigua — puede no soportar RT-DETR. Reinicia el runtime.')
else:
    print(f'✅ Versión compatible con RT-DETR')

# Verificar que RTDETR es importable
print(f'✅ RT-DETR disponible: {RTDETR}')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 25.4 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Ultralytics 8.4.43 instalado
✅ Versión compatible con RT-DETR
✅ RT-DETR disponible: <class 'ultralytics.models.rtdetr.model.RTDETR'>


## Celda 3 — Montar Google Drive

In [ ]:
# CELDA 3 — Montar Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')

# Rutas del proyecto compartidas con YOLO11n
DRIVE_BASE = '/content/drive/MyDrive/TrafficVision/datasets'
DRIVE_RUNS = '/content/drive/MyDrive/TrafficVision/runs'
RUN_NAME   = 'rtdetr_l_combined_all'

os.makedirs(DRIVE_RUNS, exist_ok=True)

print('✅ Google Drive montado')
print(f'   Datasets: {DRIVE_BASE}')
print(f'   Runs:     {DRIVE_RUNS}')
print(f'   Run name: {RUN_NAME}')
print()
print('ℹ️  Los datasets son los MISMOS que usaste para YOLO11n.')
print('   No necesitas descargar nada nuevo.')


Mounted at /content/drive
✅ Google Drive montado
   Datasets: /content/drive/MyDrive/TrafficVision/datasets
   Runs:     /content/drive/MyDrive/TrafficVision/runs
   Run name: rtdetr_l_combined_all

ℹ️  Los datasets son los MISMOS que usaste para YOLO11n.
   No necesitas descargar nada nuevo.


## Celda 4 — Verificar datasets y estimar tiempo

In [ ]:
# CELDA 4 — Verificar datasets y estimar tiempo de entrenamiento
import os

datasets = {
    'license-plates (global)':  f'{DRIVE_BASE}/license-plates',
    'license-plates-ec-1':      f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-1',
    'license-plates-ec-2':      f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-2',
    'license-plates-ec-4':      f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-4',
}

total_train = 0
total_val   = 0
all_ok      = True

print('📂 VERIFICACIÓN DE DATASETS')

for name, path in datasets.items():
    exists = os.path.exists(path)
    if exists:
        train_path = f'{path}/train/images'
        val_path   = f'{path}/valid/images'
        n_train = len(os.listdir(train_path)) if os.path.exists(train_path) else 0
        n_val   = len(os.listdir(val_path))   if os.path.exists(val_path)   else 0
        total_train += n_train
        total_val   += n_val
        print(f'  ✅ {name}')
        print(f'     train: {n_train:,} imgs  |  val: {n_val:,} imgs')
    else:
        all_ok = False
        print(f'  ❌ {name} — NO ENCONTRADO')
        print(f'     Ruta esperada: {path}')

print(f'\n  TOTAL train: {total_train:,} imágenes')
print(f'  TOTAL val:   {total_val:,} imágenes')

# ─── Estimación RT-DETR (T4, batch=8, imgsz=640)
# RT-DETR-L es ~1.8-2× más lento por época que YOLO11n
# YOLO11n: ~2.5 seg/1000 imgs → RT-DETR-L: ~5 seg/1000 imgs (batch=8, T4)
secs_per_epoch = (total_train / 1000) * 5.0
total_mins     = (secs_per_epoch * 100) / 60
print(f'\n⏱️  Estimación RT-DETR-L para 100 épocas en T4 (batch=8):')
print(f'   ~{secs_per_epoch:.0f} seg/época  →  ~{total_mins:.0f} min totales ({total_mins/60:.1f} h)')
print(f'   Colab Free: máx ~4-5h por sesión.')

if total_mins > 240:
    safe_epochs = int((240 * 60) / secs_per_epoch)
    print(f'   ⚠️  Con este dataset, una sesión T4 alcanza ~{safe_epochs} épocas.')
    print(f'   Usa save_period=5 y reanuda con CELDA 7.')

print()
print('   📊 Comparativa de velocidad (estimada, T4):')
print(f'      YOLO11n  batch=16: ~{(total_train/1000*2.5):.0f} seg/época')
print(f'      RT-DETR-L batch=8: ~{secs_per_epoch:.0f} seg/época  (más lento, más preciso)')

if not all_ok:
    print('\n⚠️  Algunos datasets faltan. Verifica que estén en Drive.')


📂 VERIFICACIÓN DE DATASETS
  ✅ license-plates (global)
     train: 7,057 imgs  |  val: 2,048 imgs
  ✅ license-plates-ec-1
     train: 54 imgs  |  val: 42 imgs
  ✅ license-plates-ec-2
     train: 90 imgs  |  val: 5 imgs
  ✅ license-plates-ec-4
     train: 375 imgs  |  val: 34 imgs

  TOTAL train: 7,576 imágenes
  TOTAL val:   2,129 imágenes

⏱️  Estimación RT-DETR-L para 100 épocas en T4 (batch=8):
   ~38 seg/época  →  ~63 min totales (1.1 h)
   Colab Free: máx ~4-5h por sesión.

   📊 Comparativa de velocidad (estimada, T4):
      YOLO11n  batch=16: ~19 seg/época
      RT-DETR-L batch=8: ~38 seg/época  (más lento, más preciso)


## Celda 5 — Crear YAML de datos

In [ ]:
# CELDA 5 — Crear data_rtdetr_combined_all.yaml
# RT-DETR usa exactamente el mismo formato YAML que YOLO
import yaml

data = {
    'train': [
        f'{DRIVE_BASE}/license-plates/train/images',
        f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-1/train/images',
        f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-2/train/images',
        f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-4/train/images',
    ],
    'val':  f'{DRIVE_BASE}/license-plates/valid/images',
    'test': f'{DRIVE_BASE}/license-plates/test/images',
    'nc':   1,
    'names': ['license plate'],
}

YAML_PATH = '/content/data_rtdetr_combined_all.yaml'
with open(YAML_PATH, 'w') as f:
    yaml.dump(data, f, default_flow_style=False)

print(f'✅ Archivo YAML generado en: {YAML_PATH}')

✅ data_rtdetr_combined_all.yaml creado
   Ruta: /content/data_rtdetr_combined_all.yaml
   Clases: 1 → ['license plate']
   Carpetas train: 4
     datasets/license-plates: 7057 imgs
     license-plates-ec-combined/license-plates-ec-1: 54 imgs
     license-plates-ec-combined/license-plates-ec-2: 90 imgs
     license-plates-ec-combined/license-plates-ec-4: 375 imgs

🏷️  Verificando labels (.txt)...
     ✅ datasets/license-plates: 7057 labels
     ✅ license-plates-ec-combined/license-plates-ec-1: 54 labels
     ✅ license-plates-ec-combined/license-plates-ec-2: 90 labels
     ✅ license-plates-ec-combined/license-plates-ec-4: 375 labels

ℹ️  RT-DETR usa el mismo formato de labels YOLO (COCO normalizado).
   No necesitas convertir los datasets.


## Celda 6 — ENTRENAR RT-DETR-L (primera vez)

> ⚠️ Solo ejecutar si NO existe un checkpoint previo.  
> Si el entrenamiento se interrumpió → usa **CELDA 7** (Reanudar).

### Diferencias clave vs YOLO11n
| Parámetro | YOLO11n | RT-DETR-L | Motivo |
|-----------|---------|-----------|--------|
| `batch` | 16 | 8 | Transformer más pesado en VRAM |
| `imgsz` | 640 | 640 | Igual — estándar para detección |
| `optimizer` | SGD (auto) | AdamW | RT-DETR converge mejor con AdamW |
| `lr0` | 0.01 | 0.0001 | LR bajo — transformers sensibles al LR |
| `warmup_epochs` | 3 | 5 | Más warmup por arquitectura más compleja |
| `cos_lr` | True | True | Cosine decay — igual de beneficioso |
| `cache` | disk | disk | Mismo cuello de botella: Drive |


In [ ]:
# CELDA 6 — ENTRENAR RT-DETR-L (primera vez)
# ⚠️  Solo ejecutar si NO existe un checkpoint previo.
#     Si el entrenamiento se interrumpió → usa CELDA 7.
import os, time
from ultralytics import RTDETR

# Verificar que no haya checkpoint previo
checkpoint = f'{DRIVE_RUNS}/{RUN_NAME}/weights/last.pt'
if os.path.exists(checkpoint):
    size_mb = os.path.getsize(checkpoint) / 1024**2
    print(f'⚠️  Ya existe un checkpoint: {checkpoint} ({size_mb:.1f} MB)')
    print('   Si quieres REANUDAR → usa CELDA 7')
    print('   Si quieres empezar DE CERO → cambia RUN_NAME arriba o borra la carpeta')
    raise SystemExit('Checkpoint existente — usa CELDA 7 para reanudar.')

print('🚀 Iniciando entrenamiento RT-DETR-L...')
print(f'   Dataset:  {YAML_PATH}')
print(f'   Destino:  {DRIVE_RUNS}/{RUN_NAME}')
print()

# Cargar RT-DETR-L preentrenado en COCO
model = RTDETR('rtdetr-l.pt')

# ─── Parámetros optimizados para RT-DETR-L en T4 con Google Drive
#
# batch=8:          Balance VRAM/velocidad en T4 (14.9 GB).
#                   Si da OOM → bajar a 4.
# imgsz=640:        Estándar; RT-DETR puede manejar 640 con L bien.
# optimizer=AdamW:  RT-DETR fue diseñado con AdamW — mejor convergencia.
# lr0=0.0001:       LR base bajo; transformers son muy sensibles al LR.
# lrf=0.01:         LR final = lr0 × lrf = 0.000001 (decay agresivo al final).
# weight_decay=1e-4: L2 regularization estándar para transformers.
# warmup_epochs=5:  Más warmup que YOLO — la atención tarda en estabilizarse.
# cos_lr=True:      Cosine LR schedule — convergencia más suave.
# cache='disk':     Evita re-leer desde Drive cada época (crítico).
# workers=2:        Drive es el cuello de botella.
# save_period=5:    Checkpoint cada 5 épocas — recuperación granular.
# patience=20:      Early stopping permisivo (dataset mixto).
# amp=True:         Mixed precision → 30% más rápido, menos VRAM.

results = model.train(
    data          = YAML_PATH,
    epochs        = 100,
    imgsz         = 640,
    batch         = 8,           # ← si OOM → bajar a 4
    name          = RUN_NAME,
    project       = DRIVE_RUNS,
    optimizer     = 'AdamW',     # ← clave para RT-DETR
    lr0           = 0.0001,      # ← LR bajo para transformers
    lrf           = 0.01,
    weight_decay  = 1e-4,
    warmup_epochs = 5,           # ← más warmup que YOLO
    patience      = 20,
    save          = True,
    save_period   = 5,
    plots         = True,
    device        = 0,
    amp           = True,
    cos_lr        = True,
    cache         = 'disk',
    workers       = 2,
    resume        = False,
    verbose       = True,
)

elapsed = (time.time() - SESSION_START) / 60
print(f'\n✅ Entrenamiento RT-DETR-L completado en {elapsed:.1f} min')
print(f'   Modelo guardado en: {DRIVE_RUNS}/{RUN_NAME}/weights/best.pt')


🚀 Iniciando entrenamiento RT-DETR-L...
   Dataset:  /content/data_rtdetr_combined_all.yaml
   Destino:  /content/drive/MyDrive/TrafficVision/runs/rtdetr_l_combined_all

Ultralytics 8.4.43 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=disk, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/data_rtdetr_combined_all.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=tr

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      1/100      6.45G     0.6434      1.321     0.2315          9        640: 100% ━━━━━━━━━━━━ 947/947 2.4s/it 38:07
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 128/128 2.8it/s 45.2s
                   all       2048       2195      0.103      0.358     0.0931     0.0426

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      2/100      6.87G     0.3857      1.359     0.1533          9        640: 0% ──────────── 0/947  0.7s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      2/100      6.87G      0.451      1.271      0.145         14        640: 100% ━━━━━━━━━━━━ 947/947 1.5it/s 10:30
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 128/128 3.0it/s 43.2s
                   all       2048       2195     0.0644      0.797      0.156     0.0968

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      3/100      6.87G     0.5194      1.189     0.1039         14        640: 0% ──────────── 0/947  0.7s

...

     20/100      6.92G     0.3379     0.3779     0.1118         23        640: 100% ━━━━━━━━━━━━ 947/947 1.5it/s 10:16
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 128/128 3.1it/s 41.3s
                   all       2048       2195      0.989      0.949      0.974      0.691

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
     21/100      6.92G     0.3529     0.3594    0.07305         13        640: 0% ──────────── 0/947  0.7s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     21/100      6.92G      0.333     0.3776     0.1106         16        640: 100% ━━━━━━━━━━━━ 947/947 1.6it/s 10:10
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 128/128 3.1it/s 41.6s
                   all       2048       2195      0.976      0.952      0.973      0.685

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
     22/100      6.92G     0.2646     0.3205    0.07493         12        640: 0% ──────────── 0/947  0.7s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     22/100      6.92G     0.3265     0.3784     0.1127         15        640: 34% ━━━━──────── 321/947 1.5it/s 3:33<7:05

## Celda 7B — ENTRENAR solo datasets Ecuador (primera vez)

> Equivalente a CELDA 7B del notebook YOLO11n.  
> Entrena un modelo especializado **solo con imágenes ecuatorianas**.  
> ⚠️ Solo ejecutar si NO existe un checkpoint Ecuador previo.  
> Si se interrumpió → usa **CELDA 7C**.

Útil para comparar si el modelo especializado en Ecuador supera al `combined_all`.


In [ ]:
# CELDA 7B — ENTRENAR RT-DETR-L solo datasets Ecuador (primera vez)
# ⚠️  Solo ejecutar si NO existe un checkpoint Ecuador previo.
#     Si el entrenamiento se interrumpió → usa CELDA 7C.
import yaml, os, time
from ultralytics import RTDETR

RUN_NAME_EC = 'rtdetr_l_ecuador_combined'

# Verificar que no haya checkpoint previo
checkpoint_ec = f'{DRIVE_RUNS}/{RUN_NAME_EC}/weights/last.pt'
if os.path.exists(checkpoint_ec):
    size_mb = os.path.getsize(checkpoint_ec) / 1024**2
    print(f'⚠️  Ya existe un checkpoint: {checkpoint_ec} ({size_mb:.1f} MB)')
    print('   Si quieres REANUDAR → usa CELDA 7C')
    print('   Si quieres empezar DE CERO → borra la carpeta manualmente en Drive')
    raise SystemExit('Checkpoint existente — usa CELDA 7C para reanudar.')

# ── Crear YAML solo Ecuador ─────────────────────────────────────────────────
data_ec = {
    'train': [
        f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-1/train/images',
        f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-2/train/images',
        f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-4/train/images',
    ],
    # ec-1 tiene val/test propios; ec-2 y ec-4 no siempre tienen val
    'val':  f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-1/valid/images',
    'test': f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-1/test/images',
    'nc':   1,
    'names': ['license plate'],
}

YAML_EC = '/content/data_rtdetr_ecuador.yaml'
with open(YAML_EC, 'w') as f:
    yaml.dump(data_ec, f, default_flow_style=False, allow_unicode=True)

# ── Verificar datasets Ecuador ──────────────────────────────────────────────
print('📂 Verificando datasets Ecuador...')
total_ec = 0
for p in data_ec['train']:
    if os.path.exists(p):
        n = len(os.listdir(p))
        total_ec += n
        label = p.split('/')[-3]
        print(f'   ✅ {label}: {n} imgs')
    else:
        print(f'   ❌ NO ENCONTRADO: {p}')

print(f'   Total train Ecuador: {total_ec} imágenes')

# Estimación de tiempo (T4, batch=8, ~5 seg/1000 imgs para RT-DETR-L)
secs_ec  = (total_ec / 1000) * 5.0
mins_ec  = (secs_ec * 100) / 60
print(f'   ⏱️  Estimación 100 épocas RT-DETR-L: ~{mins_ec:.0f} min (~{mins_ec/60:.1f} h)')
if mins_ec < 240:
    print(f'   ✅ Cabe en una sola sesión T4 de Colab Free')
else:
    safe_ep = int((240 * 60) / secs_ec)
    print(f'   ⚠️  Solo alcanzarán ~{safe_ep} épocas por sesión — usa save_period=5')

print(f'\n🚀 Iniciando entrenamiento {RUN_NAME_EC}...')
print(f'   Dataset: {YAML_EC}')
print(f'   Destino: {DRIVE_RUNS}/{RUN_NAME_EC}')
print()

model_ec = RTDETR('rtdetr-l.pt')

# ─── Mismos hiperparámetros que combined_all
# Con menos datos (solo Ecuador), patience más alto ayuda a no cortar pronto.
# lr0 igual de bajo — transformers siguen siendo sensibles al LR.
results_ec = model_ec.train(
    data          = YAML_EC,
    epochs        = 100,
    imgsz         = 640,
    batch         = 8,           # ← si OOM → bajar a 4
    name          = RUN_NAME_EC,
    project       = DRIVE_RUNS,
    optimizer     = 'AdamW',
    lr0           = 0.0001,
    lrf           = 0.01,
    weight_decay  = 1e-4,
    warmup_epochs = 5,
    patience      = 30,          # ← más permisivo: dataset Ecuador es pequeño
    save          = True,
    save_period   = 5,
    plots         = True,
    device        = 0,
    amp           = True,
    cos_lr        = True,
    cache         = 'disk',
    workers       = 2,
    resume        = False,
    verbose       = True,
)

elapsed = (time.time() - SESSION_START) / 60
print(f'\n✅ Entrenamiento Ecuador RT-DETR-L completado en {elapsed:.1f} min')
print(f'   Modelo: {DRIVE_RUNS}/{RUN_NAME_EC}/weights/best.pt')
print(f'\n📊 Comparativa disponible en CELDA 8 (evalúa ambos modelos)')


## Celda 7 — REANUDAR entrenamiento interrumpido

> Usar cuando se desconectó Colab o se agotó el tiempo de GPU.  
> **Ejecuta celdas 0 → 5 antes de esta.**

**Optimización incluida:** copia el dataset a SSD local antes de reanudar.  
El escaneo desde Drive tarda ~80 min; desde SSD local < 30 seg.


In [ ]:
# CELDA 7 — REANUDAR entrenamiento RT-DETR interrumpido
# Ejecuta celdas 0-5 antes!
import glob, os, shutil, time, yaml
from ultralytics import RTDETR

# ── Rutas ──────────────────────────────────────────────────────────────────
DRIVE_DATASET_GLOBAL = f'{DRIVE_BASE}/license-plates'
DRIVE_DATASET_EC     = f'{DRIVE_BASE}/license-plates-ec-combined'
LOCAL_BASE           = '/content/datasets'
LOCAL_GLOBAL         = f'{LOCAL_BASE}/license-plates'
LOCAL_EC             = f'{LOCAL_BASE}/license-plates-ec-combined'
LOCAL_YAML           = '/content/data_rtdetr_combined_all_local.yaml'

# ── 1. Copiar datasets a SSD local ──────────────────────────────────────────
os.makedirs(LOCAL_BASE, exist_ok=True)

for src, dst, label in [
    (DRIVE_DATASET_GLOBAL, LOCAL_GLOBAL, 'license-plates (global)'),
    (DRIVE_DATASET_EC,     LOCAL_EC,     'license-plates-ec-combined'),
]:
    if os.path.exists(dst):
        print(f'✅ {label} ya está en local — omitiendo copia')
    else:
        print(f'📦 Copiando {label}... (puede tardar 3-8 min la primera vez)')
        t0 = time.time()
        shutil.copytree(src, dst)
        mins = (time.time() - t0) / 60
        n = sum(len(files) for _, _, files in os.walk(dst))
        print(f'   ✅ {n:,} archivos copiados en {mins:.1f} min  →  {dst}')

# ── 2. Crear YAML local ─────────────────────────────────────────────────────
with open(YAML_PATH, 'r') as f:
    cfg = yaml.safe_load(f)

def local_path(p):
    return p.replace(DRIVE_BASE, LOCAL_BASE) if isinstance(p, str) else p

cfg['train'] = [local_path(p) for p in cfg['train']] if isinstance(cfg.get('train'), list) else local_path(cfg.get('train'))
cfg['val']   = local_path(cfg.get('val', ''))
cfg['test']  = local_path(cfg.get('test', ''))

with open(LOCAL_YAML, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False, allow_unicode=True)
print(f'\n📄 YAML local creado: {LOCAL_YAML}')

# ── 3. Buscar checkpoint (tolerando sufijos -2/-3/-4) ──────────────────────
def encontrar_last_pt(drive_runs, run_name):
    """Retorna la ruta a last.pt tolerando sufijos numéricos en el run."""
    exacto = f'{drive_runs}/{run_name}/weights/last.pt'
    if os.path.exists(exacto):
        return exacto, run_name

    variantes = sorted(
        glob.glob(f'{drive_runs}/{run_name}-*/weights/last.pt'),
        reverse=True
    )
    if variantes:
        run_real      = variantes[0].split('/weights/')[0]
        run_name_real = os.path.basename(run_real)
        print(f'ℹ️  Run con sufijo detectado: {run_name_real}')
        return variantes[0], run_name_real

    cualquier = sorted(
        glob.glob(f'{drive_runs}/{run_name}*/weights/*.pt'),
        key=os.path.getmtime, reverse=True
    )
    if cualquier:
        run_real      = cualquier[0].split('/weights/')[0]
        run_name_real = os.path.basename(run_real)
        print(f'ℹ️  last.pt no encontrado, usando más reciente: {cualquier[0]}')
        return cualquier[0], run_name_real

    return None, None

last_pt, run_name_real = encontrar_last_pt(DRIVE_RUNS, RUN_NAME)

if last_pt is None:
    print(f'❌ No se encontró ningún checkpoint para "{RUN_NAME}" en:')
    print(f'   {DRIVE_RUNS}/')
    print()
    print('   Si no hay checkpoint → ejecuta CELDA 6 para comenzar de cero.')
    raise FileNotFoundError('Sin checkpoint para reanudar.')

size_mb = os.path.getsize(last_pt) / 1024**2
print(f'\n✅ Checkpoint: {last_pt}  ({size_mb:.1f} MB)')

try:
    import torch
    ckpt  = torch.load(last_pt, map_location='cpu', weights_only=False)
    epoch = ckpt.get('epoch', '?')
    print(f'   Última época guardada: {epoch}/100')
    del ckpt
except Exception:
    print('   (No se pudo leer la época del checkpoint)')

# ── 4. Reanudar entrenamiento ────────────────────────────────────────────────
print('\n🔁 Reanudando entrenamiento RT-DETR-L...')

model   = RTDETR(last_pt)
results = model.train(
    data          = LOCAL_YAML,     # ← SSD local, no Drive
    epochs        = 100,
    imgsz         = 640,
    batch         = 8,
    name          = run_name_real,
    project       = DRIVE_RUNS,    # ← checkpoints siguen en Drive
    exist_ok      = True,
    optimizer     = 'AdamW',
    lr0           = 0.0001,
    lrf           = 0.01,
    weight_decay  = 1e-4,
    warmup_epochs = 5,
    patience      = 20,
    save          = True,
    save_period   = 5,
    plots         = True,
    device        = 0,
    amp           = True,
    cos_lr        = True,
    cache         = 'ram',          # ← RAM cuando imágenes en SSD local
    workers       = 4,              # ← sin cuello de botella Drive
    resume        = True,
    verbose       = True,
)

print('\n✅ Entrenamiento RT-DETR-L reanudado y completado')


## Celda 7C — REANUDAR entrenamiento Ecuador interrumpido

> Equivalente a CELDA 7C del notebook YOLO11n.  
> Usar cuando se desconectó Colab durante el entrenamiento de Ecuador.  
> **Ejecuta celdas 0 → 5 antes de esta.**


In [ ]:
# CELDA 7C — REANUDAR entrenamiento RT-DETR Ecuador interrumpido
import yaml, glob, os, shutil, time
from ultralytics import RTDETR

RUN_NAME_EC = 'rtdetr_l_ecuador_combined'

# ── Recrear YAML Ecuador si el runtime se reinició ─────────────────────────
YAML_EC = '/content/data_rtdetr_ecuador.yaml'
if not os.path.exists(YAML_EC):
    data_ec = {
        'train': [
            f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-1/train/images',
            f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-2/train/images',
            f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-4/train/images',
        ],
        'val':  f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-1/valid/images',
        'test': f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-1/test/images',
        'nc':   1,
        'names': ['license plate'],
    }
    with open(YAML_EC, 'w') as f:
        yaml.dump(data_ec, f, default_flow_style=False, allow_unicode=True)
    print(f'✅ YAML Ecuador recreado: {YAML_EC}')
else:
    print(f'✅ YAML Ecuador ya existe: {YAML_EC}')

# ── Copiar dataset Ecuador a SSD local (evita cuello de botella Drive) ──────
LOCAL_BASE = '/content/datasets'
LOCAL_EC   = f'{LOCAL_BASE}/license-plates-ec-combined'
DRIVE_EC   = f'{DRIVE_BASE}/license-plates-ec-combined'

os.makedirs(LOCAL_BASE, exist_ok=True)
if os.path.exists(LOCAL_EC):
    print(f'✅ Dataset Ecuador ya está en local — omitiendo copia')
else:
    print(f'📦 Copiando dataset Ecuador a SSD local... (2-4 min)')
    t0 = time.time()
    shutil.copytree(DRIVE_EC, LOCAL_EC)
    mins = (time.time() - t0) / 60
    n = sum(len(files) for _, _, files in os.walk(LOCAL_EC))
    print(f'   ✅ {n:,} archivos copiados en {mins:.1f} min')

# Actualizar YAML para apuntar a local
import yaml as _yaml
with open(YAML_EC, 'r') as f:
    cfg = _yaml.safe_load(f)

def local_path(p):
    return p.replace(DRIVE_BASE, LOCAL_BASE) if isinstance(p, str) else p

cfg['train'] = [local_path(p) for p in cfg['train']] if isinstance(cfg.get('train'), list) else local_path(cfg.get('train'))
cfg['val']   = local_path(cfg.get('val', ''))
cfg['test']  = local_path(cfg.get('test', ''))

YAML_EC_LOCAL = '/content/data_rtdetr_ecuador_local.yaml'
with open(YAML_EC_LOCAL, 'w') as f:
    _yaml.dump(cfg, f, default_flow_style=False, allow_unicode=True)
print(f'✅ YAML Ecuador local creado: {YAML_EC_LOCAL}')

# ── Buscar checkpoint Ecuador ───────────────────────────────────────────────
def encontrar_last_pt(drive_runs, run_name):
    exacto = f'{drive_runs}/{run_name}/weights/last.pt'
    if os.path.exists(exacto):
        return exacto, run_name
    variantes = sorted(
        glob.glob(f'{drive_runs}/{run_name}-*/weights/last.pt'), reverse=True
    )
    if variantes:
        run_real = variantes[0].split('/weights/')[0]
        run_name_real = os.path.basename(run_real)
        print(f'ℹ️  Run con sufijo detectado: {run_name_real}')
        return variantes[0], run_name_real
    cualquier = sorted(
        glob.glob(f'{drive_runs}/{run_name}*/weights/*.pt'),
        key=os.path.getmtime, reverse=True
    )
    if cualquier:
        run_real = cualquier[0].split('/weights/')[0]
        return cualquier[0], os.path.basename(run_real)
    return None, None

last_pt_ec, run_name_ec_real = encontrar_last_pt(DRIVE_RUNS, RUN_NAME_EC)

if last_pt_ec is None:
    print(f'❌ No hay checkpoint en {DRIVE_RUNS}/{RUN_NAME_EC}/')
    print('   Si no hubo ningún save → ejecuta CELDA 7B para comenzar de cero.')
    raise FileNotFoundError('Sin checkpoint Ecuador para reanudar.')

size_mb = os.path.getsize(last_pt_ec) / 1024**2
print(f'\n✅ Checkpoint: {last_pt_ec} ({size_mb:.1f} MB)')

try:
    import torch
    ckpt  = torch.load(last_pt_ec, map_location='cpu', weights_only=False)
    epoch = ckpt.get('epoch', '?')
    print(f'   Última época guardada: {epoch}/100')
    del ckpt
except Exception:
    print('   (No se pudo leer la época del checkpoint)')

# ── Reanudar ────────────────────────────────────────────────────────────────
print('\n🔁 Reanudando entrenamiento RT-DETR-L Ecuador...')

model_ec = RTDETR(last_pt_ec)
results_ec = model_ec.train(
    data          = YAML_EC_LOCAL,  # ← SSD local
    epochs        = 100,
    imgsz         = 640,
    batch         = 8,
    name          = run_name_ec_real,
    project       = DRIVE_RUNS,
    exist_ok      = True,
    optimizer     = 'AdamW',
    lr0           = 0.0001,
    lrf           = 0.01,
    weight_decay  = 1e-4,
    warmup_epochs = 5,
    patience      = 30,
    save          = True,
    save_period   = 5,
    plots         = True,
    device        = 0,
    amp           = True,
    cos_lr        = True,
    cache         = 'ram',          # ← RAM cuando imágenes en SSD local
    workers       = 4,
    resume        = True,
    verbose       = True,
)

print('\n✅ Entrenamiento Ecuador RT-DETR-L reanudado y completado')
print(f'   Modelo: {DRIVE_RUNS}/{run_name_ec_real}/weights/best.pt')


## Celda 8 — Evaluar métricas y comparar con YOLO11n

Compara RT-DETR-L contra el mejor modelo YOLO11n que ya tienes entrenado.


In [ ]:
# CELDA 8 — Evaluar métricas (RT-DETR combined_all vs RT-DETR Ecuador vs YOLO11n)
import glob, os, yaml
from ultralytics import RTDETR, YOLO

# ── Recrear YAMLs si el runtime se reinició ─────────────────────────────────
YAML_PATH = '/content/data_rtdetr_combined_all.yaml'
if not os.path.exists(YAML_PATH):
    data_all = {
        'train': [
            f'{DRIVE_BASE}/license-plates/train/images',
            f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-1/train/images',
            f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-2/train/images',
            f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-4/train/images',
        ],
        'val':  f'{DRIVE_BASE}/license-plates/valid/images',
        'test': f'{DRIVE_BASE}/license-plates/test/images',
        'nc': 1, 'names': ['license plate'],
    }
    with open(YAML_PATH, 'w') as f:
        yaml.dump(data_all, f, default_flow_style=False, allow_unicode=True)
    print(f'♻️  YAML combined_all recreado')

YAML_EC = '/content/data_rtdetr_ecuador.yaml'
if not os.path.exists(YAML_EC):
    data_ec = {
        'train': [
            f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-1/train/images',
            f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-2/train/images',
            f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-4/train/images',
        ],
        'val':  f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-1/valid/images',
        'test': f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-1/test/images',
        'nc': 1, 'names': ['license plate'],
    }
    with open(YAML_EC, 'w') as f:
        yaml.dump(data_ec, f, default_flow_style=False, allow_unicode=True)
    print(f'♻️  YAML Ecuador recreado')

# ── Función: busca run aunque tenga sufijo (-2/-3/-4) ───────────────────────
def encontrar_best_pt(drive_runs, run_name):
    exacto = f'{drive_runs}/{run_name}/weights/best.pt'
    if os.path.exists(exacto):
        return exacto, run_name
    variantes = sorted(glob.glob(f'{drive_runs}/{run_name}-*/weights/best.pt'), reverse=True)
    if variantes:
        run_real = '/'.join(variantes[0].split('/')[:-2])
        return variantes[0], os.path.basename(run_real)
    return None, None

def evaluar(best_pt, yaml_path, label, es_rtdetr=True):
    if not best_pt or not os.path.exists(best_pt):
        print(f'   ⚠️  {label}: no encontrado')
        return None
    size_mb = os.path.getsize(best_pt) / 1024**2
    print(f'\n✅ Evaluando {label}: {best_pt} ({size_mb:.1f} MB)')
    if es_rtdetr:
        model = RTDETR(best_pt)
        return model.val(data=yaml_path, imgsz=640, device=0, batch=8, plots=True)
    else:
        model = YOLO(best_pt)
        return model.val(data=yaml_path, imgsz=640, device=0, batch=16, plots=False)

# ── 1. RT-DETR-L combined_all ────────────────────────────────────────────────
best_combined, run_combined = encontrar_best_pt(DRIVE_RUNS, 'rtdetr_l_combined_all')
if run_combined and run_combined != 'rtdetr_l_combined_all':
    print(f'ℹ️  Run combined con sufijo: {run_combined}')
m_combined = evaluar(best_combined, YAML_PATH, f'RT-DETR-L combined [{run_combined}]', es_rtdetr=True)

# ── 2. RT-DETR-L solo Ecuador ────────────────────────────────────────────────
best_ec, run_ec = encontrar_best_pt(DRIVE_RUNS, 'rtdetr_l_ecuador_combined')
if run_ec and run_ec != 'rtdetr_l_ecuador_combined':
    print(f'ℹ️  Run Ecuador con sufijo: {run_ec}')
m_ecuador = evaluar(best_ec, YAML_EC, f'RT-DETR-L Ecuador [{run_ec}]', es_rtdetr=True) if best_ec else None
if not best_ec:
    print('\nℹ️  Modelo Ecuador RT-DETR no encontrado — solo se muestra combined_all.')

# ── 3. YOLO11n combined_all (referencia) ────────────────────────────────────
yolo_nombres = ['yolo11n_combined_all', 'best_ecuador_yolo11']
best_yolo, run_yolo = None, None
for nombre in yolo_nombres:
    best_yolo, run_yolo = encontrar_best_pt(DRIVE_RUNS, nombre)
    if best_yolo:
        break
m_yolo = evaluar(best_yolo, YAML_PATH, f'YOLO11n [{run_yolo}]', es_rtdetr=False) if best_yolo else None
if not best_yolo:
    print('\nℹ️  Modelo YOLO11n no encontrado — comparativa parcial.')

# ── Tabla comparativa ────────────────────────────────────────────────────────
print('\n')
print('╔═════════════════════════════════════════════════════════════════════════════╗')
print('║      COMPARATIVA: RT-DETR-L (combined / Ecuador) vs YOLO11n               ║')
print('╠═══════════════════════╦══════════════════╦════════════════╦════════════════╣')
print('║ Métrica               ║ RT-DETR combined ║ RT-DETR Ecuador║ YOLO11n        ║')
print('╠═══════════════════════╬══════════════════╬════════════════╬════════════════╣')

def fmt(val):
    return f'{val:.3f} ({val*100:.1f}%)' if val is not None else 'N/A           '

rows = [
    ('mAP@50',
     m_combined.box.map50 if m_combined else None,
     m_ecuador.box.map50  if m_ecuador  else None,
     m_yolo.box.map50     if m_yolo     else None),
    ('mAP@50-95',
     m_combined.box.map   if m_combined else None,
     m_ecuador.box.map    if m_ecuador  else None,
     m_yolo.box.map       if m_yolo     else None),
    ('Precisión',
     m_combined.box.mp    if m_combined else None,
     m_ecuador.box.mp     if m_ecuador  else None,
     m_yolo.box.mp        if m_yolo     else None),
    ('Recall',
     m_combined.box.mr    if m_combined else None,
     m_ecuador.box.mr     if m_ecuador  else None,
     m_yolo.box.mr        if m_yolo     else None),
]
for nombre, vc, ve, vy in rows:
    print(f'║ {nombre:<21s} ║ {fmt(vc):<16s} ║ {fmt(ve):<14s} ║ {fmt(vy):<14s} ║')
print('╚═══════════════════════╩══════════════════╩════════════════╩════════════════╝')

# ── Conclusión automática ────────────────────────────────────────────────────
resultados = [
    (m_combined, 'RT-DETR combined_all'),
    (m_ecuador,  'RT-DETR Ecuador'),
    (m_yolo,     'YOLO11n'),
]
resultados_validos = [(m, n) for m, n in resultados if m is not None]

if len(resultados_validos) >= 2:
    mejor_m, mejor_n = max(resultados_validos, key=lambda x: x[0].box.map50)
    print(f'\n🏆 Mejor modelo: {mejor_n} (mAP@50 = {mejor_m.box.map50*100:.1f}%)')

    if m_combined and m_ecuador:
        diff_ec = (m_ecuador.box.map50 - m_combined.box.map50) * 100
        if diff_ec > 2:
            print(f'   → RT-DETR Ecuador supera al combined en {diff_ec:+.1f} pp')
            print('     El modelo especializado se adapta mejor a placas locales.')
        elif diff_ec < -2:
            print(f'   → RT-DETR combined supera al Ecuador en {-diff_ec:+.1f} pp')
            print('     El dataset global aporta generalización importante.')
        else:
            print(f'   → Combined y Ecuador similares ({diff_ec:+.1f} pp) — usa combined_all')
            print('     (mejor generalización al incluir más datos).')

    if m_combined and m_yolo:
        diff_yolo = (m_combined.box.map50 - m_yolo.box.map50) * 100
        if diff_yolo > 2:
            print(f'   → RT-DETR-L supera a YOLO11n en {diff_yolo:+.1f} pp mAP@50')
            print('     Recomienda RT-DETR para el backend si la latencia lo permite.')
        elif diff_yolo < -2:
            print(f'   → YOLO11n supera a RT-DETR-L en {-diff_yolo:+.1f} pp mAP@50')
            print('     Considera entrenar más épocas RT-DETR o ajustar el LR.')
        else:
            print(f'   → RT-DETR y YOLO11n rinden similar ({diff_yolo:+.1f} pp).')
            print('     YOLO11n preferible si necesitas menor latencia.')

print()
print('⚡ Velocidad de inferencia estimada (GPU T4, imgsz=640, batch=1):')
print('   RT-DETR-L: ~25-35 ms/imagen  (~30-40 FPS)')
print('   YOLO11n:   ~5-8 ms/imagen    (~125-200 FPS)')


## Celda 9 — Exportar modelo para producción

Exporta a ONNX para inferencia optimizada. RT-DETR en ONNX es significativamente  
más rápido que el modelo PyTorch nativo, y compatible con OpenCV, TensorRT, etc.


In [ ]:
# CELDA 9 — Exportar RT-DETR-L para producción
import shutil, os, glob
from ultralytics import RTDETR

# ── Encontrar best.pt ────────────────────────────────────────────────────────
def encontrar_best_pt(drive_runs, run_name):
    exacto = f'{drive_runs}/{run_name}/weights/best.pt'
    if os.path.exists(exacto):
        return exacto
    variantes = sorted(glob.glob(f'{drive_runs}/{run_name}-*/weights/best.pt'), reverse=True)
    return variantes[0] if variantes else None

best_pt = encontrar_best_pt(DRIVE_RUNS, RUN_NAME)

if not best_pt:
    print(f'❌ No se encontró best.pt para {RUN_NAME}')
else:
    size_mb = os.path.getsize(best_pt) / 1024**2
    print(f'✅ Modelo encontrado: {best_pt} ({size_mb:.1f} MB)')

    # ── Copia PyTorch (.pt) a /content para descarga rápida
    export_pt  = '/content/rtdetr_l_trafficvision_best.pt'
    shutil.copy2(best_pt, export_pt)
    print(f'\n📦 Copia local: {export_pt}')

    # ── Exportar a ONNX (recomendado para producción)
    print('\n🔄 Exportando a ONNX...')
    model = RTDETR(best_pt)
    export_path = model.export(
        format  = 'onnx',
        imgsz   = 640,
        dynamic = True,   # ← permite batch size variable en inferencia
        simplify= True,   # ← simplifica el grafo ONNX (más rápido)
        opset   = 17,     # ← opset 17 soportado por TensorRT 8.6+
    )
    print(f'✅ ONNX exportado: {export_path}')

    # Copiar ONNX a Drive para no perderlo
    onnx_drive = best_pt.replace('best.pt', 'rtdetr_l_best.onnx')
    if export_path and os.path.exists(str(export_path)):
        shutil.copy2(str(export_path), onnx_drive)
        print(f'✅ ONNX guardado en Drive: {onnx_drive}')

    # ── Descarga directa desde Colab
    from google.colab import files
    print('\n📥 Descargando modelo PyTorch (.pt)...')
    files.download(export_pt)

    if export_path and os.path.exists(str(export_path)):
        print('📥 Descargando modelo ONNX...')
        files.download(str(export_path))

    # ── Resumen del run
    run_dir    = os.path.dirname(os.path.dirname(best_pt))
    results_csv = f'{run_dir}/results.csv'
    if os.path.exists(results_csv):
        import pandas as pd
        df = pd.read_csv(results_csv)
        df.columns = df.columns.str.strip()
        if 'metrics/mAP50(B)' in df.columns:
            best_row   = df.loc[df['metrics/mAP50(B)'].idxmax()]
            best_epoch = int(best_row['epoch']) + 1
            best_map50 = best_row['metrics/mAP50(B)']
            print(f'\n📊 Mejor época: {best_epoch}/100  →  mAP@50 = {best_map50:.4f} ({best_map50*100:.1f}%)')

    print()
    print('🔧 Para usar RT-DETR en el backend (plate_detector.py):')
    print()
    print('   # Opción A — PyTorch (más fácil):')
    print('   from ultralytics import RTDETR')
    print('   model = RTDETR("ml/models/trained/rtdetr_l_combined_all/best.pt")')
    print('   results = model("frame.jpg")')
    print()
    print('   # Opción B — ONNX (más rápido en producción):')
    print('   import onnxruntime as ort')
    print('   sess = ort.InferenceSession("rtdetr_l_best.onnx")')


## 💡 Guía rápida — Solución de problemas

### Si sale `CUDA out of memory`
```
# En Celda 6 o 7, cambiar:
batch = 4       # reducir de 8 → 4
imgsz = 512     # reducir de 640 → 512 si persiste el OOM
```

### Si se desconecta durante el entrenamiento
1. Abre el notebook de nuevo
2. Ejecuta celdas **0 → 5**
3. Ejecuta **CELDA 7** (Reanudar) — RT-DETR retoma desde el último `save_period`

### Señales de que el entrenamiento va bien
- `box_loss` bajando consistentemente ✅
- `giou_loss` bajando (específico de RT-DETR) ✅
- `mAP50` subiendo progresivamente ✅

### Tiempos estimados (T4, batch=8, imgsz=640)
| Épocas | Tiempo estimado |
|--------|----------------|
| 20     | ~80-100 min    |
| 50     | ~200-250 min   |
| 100    | ~6-7 h         |

> RT-DETR converge con **menos épocas** que YOLO en muchos datasets.  
> Monitorea el mAP@50 — si se estabiliza antes de la época 100, el early  
> stopping (patience=20) detendrá automáticamente el entrenamiento.

### Comparativa de modelos disponibles
| Modelo | Parámetros | VRAM (batch=8) | mAP COCO |
|--------|-----------|----------------|----------|
| rtdetr-l.pt | 32M | ~7 GB | 53.0 |
| rtdetr-x.pt | 67M | ~14 GB | 54.8 (puede dar OOM en T4) |
